<a href="https://colab.research.google.com/github/saithrisank12/Understanding-how-LLM-is-built/blob/main/Understanding_and_Building_an_LLm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, I am going to build a very small **Language Model** from scratch to understand how modern LLMs work internally.

**Step 1 — Build the Model**

Before we can train a model, we first need to build the model itself. We use PyTorch, a deep learning framework, to create its structure. In PyTorch, a model is created using a class that inherits from nn.Module. The __init__() function is used to define what our model contains. For now, we are creating only the basic model structure; it has not learned anything yet.

In [44]:
import torch
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()

model = TinyModel()

print(model)

TinyModel()


**Step 2 — Prepare the Training Data**

Now that we have created our model, we need data to train it. During training, the model will use this data to learn patterns and update its weights. When the model was first created, its weights were initialized with starting values; they are not trained or meaningful yet. So, our next step is to provide some text that the model can learn from.

In [45]:
text = """
The sun rises in the east.
The earth revolves around the sun.
The earth revolves around the sun.
The moon revolves around the earth.
Artificial intelligence learns from data.
Machine learning finds patterns in data.
"""

***Step 3 — Clean the Training Data***

We now clean the existing text before giving it to the model. This removes empty lines, unnecessary spaces, and exact duplicate lines while keeping legitimate repeated words and information.

In [46]:
lines = [line.strip() for line in text.splitlines() if line.strip()]
cleaned_lines = list(dict.fromkeys(lines))

text = "\n".join(cleaned_lines)

print(text)

The sun rises in the east.
The earth revolves around the sun.
The moon revolves around the earth.
Artificial intelligence learns from data.
Machine learning finds patterns in data.


***Step 4 — Tokenization & Vocabulary***

So far, we have collected and cleaned our text. But computers and neural networks work with numbers, not raw text. So first, we break our text into smaller units called tokens, then create a vocabulary of the unique tokens and assign each one a number (ID). This prepares our text to be processed by the model.

In [47]:
tokens = text.split()

vocab = sorted(set(tokens))

stoi = {token: i for i, token in enumerate(vocab)}

print("Tokens:")
print(tokens)

print("\nVocabulary:")
print(vocab)

print("\nToken IDs:")
print(stoi)

Tokens:
['The', 'sun', 'rises', 'in', 'the', 'east.', 'The', 'earth', 'revolves', 'around', 'the', 'sun.', 'The', 'moon', 'revolves', 'around', 'the', 'earth.', 'Artificial', 'intelligence', 'learns', 'from', 'data.', 'Machine', 'learning', 'finds', 'patterns', 'in', 'data.']

Vocabulary:
['Artificial', 'Machine', 'The', 'around', 'data.', 'earth', 'earth.', 'east.', 'finds', 'from', 'in', 'intelligence', 'learning', 'learns', 'moon', 'patterns', 'revolves', 'rises', 'sun', 'sun.', 'the']

Token IDs:
{'Artificial': 0, 'Machine': 1, 'The': 2, 'around': 3, 'data.': 4, 'earth': 5, 'earth.': 6, 'east.': 7, 'finds': 8, 'from': 9, 'in': 10, 'intelligence': 11, 'learning': 12, 'learns': 13, 'moon': 14, 'patterns': 15, 'revolves': 16, 'rises': 17, 'sun': 18, 'sun.': 19, 'the': 20}


***Step 5 — Convert the Text into Token IDs***

We already have the Token → ID mapping. Now we use it to convert every token in our original sequence into its corresponding ID. This gives us the numerical sequence that represents our actual training text.

In [48]:
token_ids = [stoi[token] for token in tokens]

print("Token IDs:")
print(token_ids)

Token IDs:
[2, 18, 17, 10, 20, 7, 2, 5, 16, 3, 20, 19, 2, 14, 16, 3, 20, 6, 0, 11, 13, 9, 4, 1, 12, 8, 15, 10, 4]


***Step 6 — Create Training Pairs***

Now that our text has been converted into token IDs, we need to create examples for the model to learn from. For a language model, the basic task is predicting the next token, so the input is the current sequence and the target is the same sequence shifted one position forward.

In [49]:
x = torch.tensor(token_ids[:-1])
y = torch.tensor(token_ids[1:])

print("Input :", x)
print("Target:", y)

Input : tensor([ 2, 18, 17, 10, 20,  7,  2,  5, 16,  3, 20, 19,  2, 14, 16,  3, 20,  6,
         0, 11, 13,  9,  4,  1, 12,  8, 15, 10])
Target: tensor([18, 17, 10, 20,  7,  2,  5, 16,  3, 20, 19,  2, 14, 16,  3, 20,  6,  0,
        11, 13,  9,  4,  1, 12,  8, 15, 10,  4])


***Step 7 — Create Embeddings***

Our text is now represented as token IDs, but these IDs are only labels. We now need to convert each token ID into a vector of numbers that the neural network can work with. An embedding layer does this conversion, and its values will later be learned and updated during training.

In [50]:
embedding = nn.Embedding(len(vocab), 16)

embedded = embedding(x)

print("Input shape:", x.shape)
print("Embedding shape:", embedded.shape)

Input shape: torch.Size([28])
Embedding shape: torch.Size([28, 16])


***Step 8 — Process the Embeddings and Make a Prediction***

We now have our text converted into embedding vectors. These vectors are passed into our neural network, which processes the numbers and tries to predict what token should come next. Since the model has not been trained yet, its prediction will not be accurate at this stage.

In [51]:
model = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, len(vocab))
)

output = model(embedded)

print("Output shape:", output.shape)

Output shape: torch.Size([28, 21])


***Step 9 — Calculate the Loss***

Our model has made a prediction, but we need to know how wrong it is. We compare the model's prediction with the correct target token we created earlier. The difference between the prediction and the correct answer is measured using loss. A higher loss means the model is more wrong; a lower loss means it is closer to the correct answer.

In [52]:
loss_fn = nn.CrossEntropyLoss()

loss = loss_fn(output, y)

print("Loss:", loss.item())

Loss: 3.056748867034912


***Step 10 — Backpropagation & Update the Weights***

We know how wrong the model is from the loss. Now we use backpropagation to calculate how the weights should change, and an Adam optimizer uses that information to update the weights. This is how the model learns from its mistake.

In [53]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

loss.backward()
optimizer.step()

print("Weights updated.")

Weights updated.


***Step 11 — Train the Model***

Our model has finished training, so now we want to see what it has learned. We give the model the input tokens again, and it predicts the next token for each position. The model does not directly output words; it outputs numbers, because every word in our vocabulary has a token ID. For example, if sun → 18, and the model predicts 18, it means the model predicted “sun”. torch.argmax() simply chooses the token with the highest prediction score.



In [54]:
for step in range(100):
    optimizer.zero_grad()

    embedded = embedding(x)
    output = model(embedded)
    loss = loss_fn(output, y)

    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

Step 0, Loss: 3.0436
Step 10, Loss: 2.9149
Step 20, Loss: 2.7877
Step 30, Loss: 2.6581
Step 40, Loss: 2.5226
Step 50, Loss: 2.3793
Step 60, Loss: 2.2254
Step 70, Loss: 2.0623
Step 80, Loss: 1.8942
Step 90, Loss: 1.7252


***Step 12 — Test the Model***

Our model has finished training, so now we want to see what it has learned. We give the model the input tokens again, and it predicts the next token for each position. The model does not directly output words; it outputs numbers, because every word in our vocabulary has a token ID. For example, if sun → 18, and the model predicts 18, it means the model predicted “sun”. torch.argmax() simply chooses the token with the highest prediction score.



In [55]:
with torch.no_grad():
    embedded = embedding(x)
    output = model(embedded)

prediction = torch.argmax(output, dim=1)

print("Predicted token IDs:")
print(prediction)

Predicted token IDs:
tensor([ 5, 17, 10, 20, 19,  2,  5, 16,  3, 20, 19,  2,  5, 16,  3, 20, 19,  0,
        11, 13,  9,  4, 19, 12,  2, 15, 10, 20])


***Step 13 — Generate Text***

Our model has learned to predict the next token. Now we will use that ability to generate new text. We start with one word, such as “The”. The model predicts the next token, then we use that predicted token as the next input. We repeat this process several times. Finally, we convert the predicted token IDs back into words so we can read what the model generated.

In [56]:
id_to_token = {i: token for token, i in stoi.items()}

current_token = stoi["The"]

generated = ["The"]

for _ in range(10):
    token = torch.tensor([current_token])

    with torch.no_grad():
        embedded = embedding(token)
        output = model(embedded)

    next_token = torch.argmax(output, dim=1).item()

    generated.append(id_to_token[next_token])
    current_token = next_token

print("Generated text:")
print(" ".join(generated))

Generated text:
The earth revolves around the sun. The earth revolves around the


***What is happening?***

"The"
  ->
Model predicts
  ->
"sun"
  ->
Model predicts
  ->
"rises"
  ->
Model predicts
  ->
"in"
  ->
continues...

***Step 14 — Conclude the Basic Language Model***

Context:
We have now completed the full learning process for our basic language model. We started with raw text, cleaned it, converted words into tokens and numbers, created embeddings, passed them through a neural network, calculated the loss, trained the model using backpropagation and Adam, tested its predictions, and finally generated text. This model is intentionally simple and does not use a Transformer, RNN, or CNN. Its purpose was to understand the basic mechanics of how a language model learns.

 ***Final Workflow — Tiny Language Model from Scratch***


                    RAW TEXT
                       │
                       ▼
                ┌─────────────┐
                │ Clean Data  │
                └─────────────┘
                       │
                       ▼
             ┌──────────────────┐
             │ Tokenization +   │
             │   Vocabulary     │
             └──────────────────┘
                       │
                       ▼
                ┌─────────────┐
                │  Token IDs   │
                └─────────────┘
                       │
                       ▼
                ┌─────────────┐
                │  Embeddings │
                └─────────────┘
                       │
                       ▼
             ┌──────────────────┐
             │ Neural Network   │
             └──────────────────┘
                       │
                       ▼
                ┌─────────────┐
                │  Prediction │
                └─────────────┘
                       │
                       ▼
                ┌─────────────┐
                │    Loss     │
                └─────────────┘
                       │
                       ▼
             ┌──────────────────┐
             │ Backpropagation  │
             └──────────────────┘
                       │
                       ▼
             ┌──────────────────┐
             │ Weight Update    │
             │     (Adam)       │
             └──────────────────┘
                       │
                       ▼
                ┌─────────────┐
                │   Training  │
                └─────────────┘
                       │
                       ▼
                ┌─────────────┐
                │    Testing  │
                └─────────────┘
                       │
                       ▼
             ┌──────────────────┐
             │  Text Generation │
             └──────────────────┘
                       │
                       ▼
                     OUTPUT